In [56]:
# =====================================
# 1) IMPORT
# =====================================
import os
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from catboost import CatBoostClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 150)

In [57]:
# =====================================
# 2) LOAD DATA
# =====================================
DATA_DIR = Path(r"D:\UI")
# DATA_DIR = Path("/kaggle/input/competitions/datathon-playground-2026")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:", train.shape)
print("test :", test.shape)
print("sub  :", sample_submission.shape)

display(train.head())

train: (3200, 43)
test : (800, 42)
sub  : (800, 2)


,id,nilai_minggu_01,nilai_minggu_02,nilai_minggu_03,nilai_minggu_04,nilai_minggu_05,nilai_minggu_06,nilai_minggu_07,nilai_minggu_08,nilai_minggu_09,nilai_minggu_10,nilai_minggu_11,nilai_minggu_12,skor_motivasi,skor_kedisiplinan,aktivitas_hari_01,aktivitas_hari_02,aktivitas_hari_03,aktivitas_hari_04,aktivitas_hari_05,aktivitas_hari_06,aktivitas_hari_07,aktivitas_hari_08,aktivitas_hari_09,aktivitas_hari_10,aktivitas_hari_11,aktivitas_hari_12,aktivitas_hari_13,aktivitas_hari_14,aktivitas_hari_15,aktivitas_hari_16,tugas_selesai,tugas_diberikan,kelas,urutan_ujian,skor_tryout,jarak_rumah_km,skor_ekstrakurikuler,indeks_kehadiran,skor_literasi,jumlah_saudara,skor_minat_belajar,target
0,0,2.6,6.4,3.6,1.0,4.4,8.5,-4.0,-0.1,-4.7,1.0,-4.7,-5.1,-0.53,-0.33,64.3,42.7,42.4,65.4,77.6,61.2,36.8,34.4,51.8,89.6,55.2,44.7,42.0,46.7,69.2,46.9,8,9,114,0.0817,72.3,0.46,0.00,-0.38,-0.24,-0.70,-0.62,3
1,1,-0.3,5.7,1.7,1.3,6.1,5.8,-0.1,-3.8,-3.1,-3.1,-1.0,1.0,-2.24,1.82,27.8,51.6,53.6,56.4,40.7,68.0,66.4,30.5,51.8,74.3,61.4,41.6,38.7,72.0,31.7,32.6,46,85,301,0.3388,41.4,-1.29,0.51,0.86,-0.70,-1.04,2.15,1
2,2,-4.8,-0.8,-3.5,-6.3,-3.7,-1.5,10.7,8.7,5.8,5.2,7.2,9.9,-1.18,-1.34,33.9,50.4,50.0,36.4,73.0,66.4,36.6,62.2,60.5,35.9,49.0,63.3,43.7,47.2,53.5,44.9,23,75,28,0.0517,73.2,-0.03,-0.33,0.14,0.80,0.26,-0.57,3
3,4,4.7,1.1,2.1,0.2,1.9,1.4,-0.5,-0.3,0.5,-1.4,-1.2,3.6,-0.60,0.69,51.0,48.0,36.5,31.8,53.0,60.7,76.7,65.1,51.1,39.0,36.4,41.9,49.1,56.2,70.6,64.5,95,110,580,0.0310,59.1,-0.20,0.24,1.68,-0.44,1.32,-0.64,1
4,5,13.3,3.5,3.4,4.1,6.2,8.7,-5.0,-6.1,-5.2,-8.2,-7.1,-6.6,-1.19,-0.67,59.2,43.0,43.6,64.6,57.7,42.2,57.7,57.7,47.9,37.2,62.3,77.4,36.5,42.2,58.5,52.8,24,32,127,0.5829,60.7,1.06,0.49,0.42,-1.36,-0.45,-1.92,3


In [58]:
# =====================================
# 3) DETEKSI TARGET & ID
# =====================================
id_col = sample_submission.columns[0]
sub_target_col = sample_submission.columns[1]

possible_targets = [c for c in train.columns if c not in test.columns]
if len(possible_targets) == 1:
    target_col = possible_targets[0]
else:
    non_id_candidates = [c for c in possible_targets if c != id_col]
    target_col = non_id_candidates[0] if len(non_id_candidates) > 0 else possible_targets[0]

print("id_col    =", id_col)
print("target_col=", target_col)
print("sub_target=", sub_target_col)

id_col    = id
target_col= target
sub_target= target


In [59]:
# =====================================
# 4) FEATURE ENGINEERING YANG LEBIH KUAT
# =====================================
def add_prefix_features(df: pd.DataFrame, prefix: str, new_name: str) -> pd.DataFrame:
    cols = [c for c in df.columns if c.startswith(prefix)]
    if len(cols) == 0:
        return df

    vals = df[cols].copy()
    num = vals.apply(pd.to_numeric, errors="coerce")

    df[f"{new_name}_count"] = num.notna().sum(axis=1)
    df[f"{new_name}_mean"] = num.mean(axis=1)
    df[f"{new_name}_std"] = num.std(axis=1)
    df[f"{new_name}_min"] = num.min(axis=1)
    df[f"{new_name}_max"] = num.max(axis=1)
    df[f"{new_name}_range"] = df[f"{new_name}_max"] - df[f"{new_name}_min"]
    df[f"{new_name}_sum"] = num.sum(axis=1)

    # trend / slope sederhana
    def slope(row):
        y = row.values.astype(float)
        mask = ~np.isnan(y)
        y = y[mask]
        if len(y) < 2:
            return 0.0
        x = np.arange(len(y))
        return np.polyfit(x, y, 1)[0]

    df[f"{new_name}_trend"] = num.apply(slope, axis=1)

    # selisih awal-akhir
    first_col = cols[0]
    last_col = cols[-1]
    df[f"{new_name}_first"] = pd.to_numeric(df[first_col], errors="coerce")
    df[f"{new_name}_last"] = pd.to_numeric(df[last_col], errors="coerce")
    df[f"{new_name}_delta"] = df[f"{new_name}_last"] - df[f"{new_name}_first"]

    return df


def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # prefix-based features
    df = add_prefix_features(df, "nilai_minggu_", "nm")
    df = add_prefix_features(df, "aktivitas_hari_", "ah")

    # fitur umum yang sangat sering membantu
    if "tugas_selesai" in df.columns and "tugas_diberikan" in df.columns:
        df["task_completion_ratio"] = df["tugas_selesai"] / (df["tugas_diberikan"] + 1e-6)
        df["task_missing"] = df["tugas_diberikan"] - df["tugas_selesai"]

    if "skor_motivasi" in df.columns and "skor_kedisiplinan" in df.columns:
        df["motivasi_x_disiplin"] = df["skor_motivasi"] * df["skor_kedisiplinan"]
        df["motivasi_plus_disiplin"] = df["skor_motivasi"] + df["skor_kedisiplinan"]
        df["motivasi_minus_disiplin"] = df["skor_motivasi"] - df["skor_kedisiplinan"]

    if "skor_tryout" in df.columns:
        if "nm_mean" in df.columns:
            df["tryout_vs_nm_mean"] = df["skor_tryout"] - df["nm_mean"]
        if "nm_last" in df.columns:
            df["tryout_vs_nm_last"] = df["skor_tryout"] - df["nm_last"]

    if "nm_trend" in df.columns:
        df["trend_abs"] = df["nm_trend"].abs()

    if "ah_trend" in df.columns:
        df["activity_trend_abs"] = df["ah_trend"].abs()

    # hitung missing value per baris
    df["missing_count"] = df.isna().sum(axis=1)

    return df

In [60]:
# =====================================
# 5) SPLIT X / y
# =====================================
X = train.drop(columns=[target_col]).copy()
y = train[target_col].copy()
X_test = test.copy()

# buang id dari fitur
X = X.drop(columns=[id_col], errors="ignore")
X_test = X_test.drop(columns=[id_col], errors="ignore")

# feature engineering
X = feature_engineering(X)
X_test = feature_engineering(X_test)

print("X shape after FE    :", X.shape)
print("X_test shape after FE:", X_test.shape)

X shape after FE    : (3200, 73)
X_test shape after FE: (800, 73)


In [61]:
# =====================================
# 6) CLEANING
# =====================================
cat_cols = [c for c in X.columns if X[c].dtype == "object" or str(X[c].dtype) == "category" or X[c].dtype == "bool"]
num_cols = [c for c in X.columns if c not in cat_cols]

for c in cat_cols:
    X[c] = X[c].astype("string").fillna("MISSING")
    X_test[c] = X_test[c].astype("string").fillna("MISSING")

for c in num_cols:
    med = X[c].median()
    X[c] = X[c].fillna(med)
    X_test[c] = X_test[c].fillna(med)

print("categorical:", len(cat_cols))
print("numerical   :", len(num_cols))

categorical: 0
numerical   : 73


In [62]:
# =====================================
# 7) ENCODE TARGET
# =====================================
le = LabelEncoder()
y_enc = le.fit_transform(y.astype(str))

print("classes:", list(le.classes_))
print(pd.Series(y_enc).value_counts().sort_index())

classes: ['0', '1', '2', '3']
0    813
1    796
2    784
3    807
Name: count, dtype: int64


In [63]:
# =====================================
# 8) CATBOOST GPU + CV
# =====================================
FAST_MODE = False   # True = cepat, False = lebih serius

n_splits = 3 if FAST_MODE else 5
iterations = 800 if FAST_MODE else 2000

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X), dtype=int)
test_pred_proba = np.zeros((len(X_test), len(le.classes_)))
scores = []

cat_features = [X.columns.get_loc(c) for c in cat_cols]

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y_enc), 1):
    X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
    y_tr, y_va = y_enc[tr_idx], y_enc[va_idx]

    model = CatBoostClassifier(
        task_type="GPU",
        devices="0",
        loss_function="MultiClass",
        eval_metric="Accuracy",
        iterations=2500,
        learning_rate=0.25,
        depth=10,
        l2_leaf_reg=8,
        random_seed=42 + fold,
        auto_class_weights="Balanced",
        od_type="Iter",
        od_wait=250,
        verbose=200,
        allow_writing_files=False
    )

    model.fit(
        X_tr, y_tr,
        cat_features=cat_features,
        eval_set=(X_va, y_va),
        use_best_model=True
    )

    va_pred = model.predict(X_va).astype(int).ravel()
    fold_score = accuracy_score(y_va, va_pred)
    scores.append(fold_score)
    oof_pred[va_idx] = va_pred

    test_pred_proba += model.predict_proba(X_test) / n_splits

    print(f"Fold {fold} Accuracy: {fold_score:.5f}")

print("\nCV mean accuracy:", np.mean(scores))
print("CV std accuracy :", np.std(scores))
print("OOF accuracy    :", accuracy_score(y_enc, oof_pred))

0:	learn: 0.5954478	test: 0.4216326	best: 0.4216326 (0)	total: 33.6ms	remaining: 1m 23s
200:	learn: 1.0000000	test: 0.5356119	best: 0.5371938 (192)	total: 7.41s	remaining: 1m 24s
400:	learn: 1.0000000	test: 0.5182227	best: 0.5401211 (226)	total: 14.8s	remaining: 1m 17s
bestTest = 0.5401211144
bestIteration = 226
Shrink model to first 227 iterations.
Fold 1 Accuracy: 0.54219
0:	learn: 0.6112722	test: 0.4022588	best: 0.4022588 (0)	total: 41.9ms	remaining: 1m 44s
200:	learn: 1.0000000	test: 0.5258820	best: 0.5449474 (112)	total: 7.99s	remaining: 1m 31s
bestTest = 0.5449473813
bestIteration = 112
Shrink model to first 113 iterations.
Fold 2 Accuracy: 0.54688
0:	learn: 0.5761492	test: 0.4236784	best: 0.4236784 (0)	total: 34.4ms	remaining: 1m 26s
200:	learn: 1.0000000	test: 0.5586936	best: 0.5586936 (200)	total: 7.24s	remaining: 1m 22s
400:	learn: 1.0000000	test: 0.5538112	best: 0.5679835 (223)	total: 14.4s	remaining: 1m 15s
bestTest = 0.5679834867
bestIteration = 223
Shrink model to first 2

In [64]:
# =====================================
# 9) ANALISIS ERROR
# =====================================
print(classification_report(y_enc, oof_pred))
print(confusion_matrix(y_enc, oof_pred))

              precision    recall  f1-score   support

           0       0.66      0.70      0.68       813
           1       0.42      0.38      0.40       796
           2       0.44      0.40      0.42       784
           3       0.65      0.72      0.68       807

    accuracy                           0.55      3200
   macro avg       0.54      0.55      0.55      3200
weighted avg       0.54      0.55      0.55      3200

[[568 183  52  10]
 [225 304 185  82]
 [ 58 190 314 222]
 [ 14  55 155 583]]


In [65]:
# =====================================
# 10) FEATURE IMPORTANCE
# =====================================
fi = pd.DataFrame({
    "feature": X.columns,
    "importance": model.get_feature_importance()
}).sort_values("importance", ascending=False)

display(fi.head(30))

,feature,importance
63,task_completion_ratio,11.174440
65,motivasi_x_disiplin,9.769609
70,trend_abs,5.678008
43,nm_std,3.354055
25,aktivitas_hari_12,2.947254
28,aktivitas_hari_15,2.543958
21,aktivitas_hari_08,2.523477
20,aktivitas_hari_07,2.405149
19,aktivitas_hari_06,2.333412
17,aktivitas_hari_04,2.255819


In [66]:
xgb_pred = np.zeros((len(X_test), len(le.classes_)))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y_enc), 1):
    X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
    y_tr, y_va = y_enc[tr_idx], y_enc[va_idx]

    xgb = XGBClassifier(
        objective="multi:softprob",
        num_class=len(le.classes_),
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42 + fold,
        tree_method="hist"
    )

    xgb.fit(X_tr, y_tr)
    xgb_pred += xgb.predict_proba(X_test) / skf.n_splits

In [67]:
final_test_proba = (cat_pred + lgbm_pred + xgb_pred) / 3
final_pred_idx = np.argmax(final_test_proba, axis=1)
final_pred = le.inverse_transform(final_pred_idx)

In [68]:
pd.Series(final_pred).value_counts()

3    245
0    224
1    172
2    159
Name: count, dtype: int64

In [69]:
# =====================================
# 11) SUBMISSION
# =====================================
final_pred_idx = np.argmax(test_pred_proba, axis=1)
final_pred = le.inverse_transform(final_pred_idx)

submission = sample_submission.copy()
submission[sub_target_col] = final_pred
submission.to_csv("submission.csv", index=False)

display(submission.head())
print("Saved: submission.csv")
print(pd.Series(final_pred).value_counts())

,id,target
0,3,0
1,12,2
2,14,2
3,18,3
4,28,0


Saved: submission.csv
3    217
0    210
2    204
1    169
Name: count, dtype: int64
